# 21. Deep Promotion Time Cure Model (Deep-PTCM) for Competing Risks

**Reference:** Medina-Olivares, V., Lessmann, S. & Klein, N. (2024). "The Deep Promotion Time Cure Model." *IEEE Trans. Neural Netw. Learn. Syst.*, 35(12), 18848-18858.

## Methodology

The **Promotion Time Cure Model (PTCM)** assumes each borrower $i$ has $K_i \sim \text{Poisson}(\theta(\mathbf{x}_i))$ unobserved latent risk factors. Borrowers with $K_i = 0$ are **cured** -- they never experience the event. The cure fraction is $\pi(\mathbf{x}) = \exp(-\theta(\mathbf{x}))$.

The **Deep-PTCM** replaces the traditional linear predictor $\log\theta = \mathbf{w}^\top\mathbf{x} + b$ with a deep neural network, optionally decomposed via **orthogonalization** into interpretable linear effects plus a nonlinear residual.

### Competing Risks Extension

We extend the single-event PTCM to competing risks (prepayment + default) via independent latent causes:

$$S(t; \mathbf{x}) = \prod_k \exp\bigl(-\theta_k(\mathbf{x})\, F_k(t)\bigr)$$

where $F_k$ is a piecewise-exponential baseline CDF for cause $k$, and $\theta_k$ is parameterized by a shared DNN with cause-specific heads.

### Key Advantages
- **Cure fractions**: Explicitly models the probability that a borrower will *never* prepay or default
- **Flexible nonlinearity**: DNN captures complex covariate interactions
- **Interpretability**: Orthogonalization separates linear from nonlinear effects

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch

sys.path.insert(0, str(Path.cwd().parent))

from src.competing_risks.deep_ptcm import (
    CompetingRisksDeepPTCM,
    fit_deep_ptcm_competing_risks,
    STATIC_FEATURES,
    TRAIN_FOLDS,
    VAL_FOLDS,
    TEST_FOLD,
    get_device,
)
from src.competing_risks.evaluation import (
    time_dependent_concordance_index,
    brier_score_competing_risks,
    auc_cure,
    integrated_brier_score,
    EVAL_TIMES,
)

from sklearn.preprocessing import StandardScaler

print(f"PyTorch {torch.__version__}")
print(f"Device: {get_device()}")

## Configuration

In [ ]:
# --- Paths ---
DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
FIGURES_DIR = Path('../figures/21_deep_ptcm')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- Features ---
FEATURE_COLS = ['int_rate', 'log_upb', 'fico_score', 'dti_r', 'ltv_r']

# --- Hyperparameters (aligned with Medina-Olivares et al. 2024) ---
PTCM_PARAMS = dict(
    num_intervals=15,
    shared_layers=[512, 512],
    head_layers=[],
    dropout=0.2,
    batch_norm=False,
    orthogonalize=False,
    lr=0.01,
    weight_decay=0.0,
    lr_decay_rate=0.75,
    lr_decay_steps=100,
    batch_size=512,
    epochs=200,
    patience=20,
    verbose=True,
    random_state=42,
)

# --- Event mapping ---
EVENT_NAMES = {0: 'Censored', 1: 'Prepay', 2: 'Default'}

print("Feature columns:", FEATURE_COLS)
print("PTCM hyperparameters:")
for k, v in PTCM_PARAMS.items():
    print(f"  {k}: {v}")

## Data Loading

In [ ]:
df = pd.read_parquet(DATA_DIR / 'blumenstock_dataset2.parquet')

# Derive log_upb
df['log_upb'] = np.log(df['orig_upb'].clip(lower=1).astype(float))

print(f"Loaded {len(df):,} loans")
print(f"Vintages: {df['vintage_year'].min()} - {df['vintage_year'].max()}")
print(f"Duration range: {df['duration'].min()} - {df['duration'].max()} months")
print(f"\nEvent distribution:")
for code, name in EVENT_NAMES.items():
    n = (df['event_code'] == code).sum()
    print(f"  {name} (k={code}): {n:,} ({100*n/len(df):.1f}%)")
print(f"\nFolds: {sorted(df['fold'].unique())}")
print(f"Missing values in features:")
for c in FEATURE_COLS:
    print(f"  {c}: {df[c].isna().sum()}")

## Train / Validation / Test Split

In [ ]:
# Drop rows with missing features
df_clean = df.dropna(subset=FEATURE_COLS).copy()
print(f"After dropping NaN: {len(df_clean):,} loans (dropped {len(df) - len(df_clean):,})")

# Split by folds (matching DeepHit / Sadhwani notebooks)
train_df = df_clean[df_clean['fold'].isin(TRAIN_FOLDS)].copy()
val_df = df_clean[df_clean['fold'].isin(VAL_FOLDS)].copy()
test_df = df_clean[df_clean['fold'] == TEST_FOLD].copy()

print(f"\nTraining set (folds {TRAIN_FOLDS}): {len(train_df):,} loans")
print(f"Validation set (fold {VAL_FOLDS}): {len(val_df):,} loans")
print(f"Test set (fold {TEST_FOLD}): {len(test_df):,} loans")

for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n{name} event distribution:")
    for code, ename in EVENT_NAMES.items():
        n = (split_df['event_code'] == code).sum()
        print(f"  {ename}: {n:,}")

## Model Training (Deep-PTCM)

In [ ]:
model = fit_deep_ptcm_competing_risks(
    df=train_df,
    feature_cols=FEATURE_COLS,
    duration_col='duration',
    event_col='event_code',
    event_types=[1, 2],
    val_df=val_df,
    **PTCM_PARAMS,
)

print(f"\nTraining complete.")
print(f"  Epochs run: {len(model.history_['train_loss'])}")
print(f"  Best val loss: {min(model.history_['val_loss']):.4f}")
print(f"  Breakpoints: {model.breakpoints_}")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 5))
epochs = range(1, len(model.history_['train_loss']) + 1)
ax.plot(epochs, model.history_['train_loss'], label='Train NLL')
ax.plot(epochs, model.history_['val_loss'], label='Val NLL')
ax.set_xlabel('Epoch')
ax.set_ylabel('Negative Log-Likelihood')
ax.set_title('Deep-PTCM Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Cure Fraction Analysis

The cure fraction $\pi_k(\mathbf{x}) = \exp(-\theta_k(\mathbf{x}))$ gives the probability that borrower $\mathbf{x}$ will **never** experience event $k$.

In [ ]:
# Compute cure fractions on test set
cure = model.predict_cure_fraction(test_df[FEATURE_COLS])

print("Cure fraction statistics (test set):")
for name, values in cure.items():
    print(f"\n  {name}:")
    print(f"    Mean:   {values.mean():.4f}")
    print(f"    Median: {np.median(values):.4f}")
    print(f"    Min:    {values.min():.4f}")
    print(f"    Max:    {values.max():.4f}")

# Compare to observed rates
n_test = len(test_df)
obs_cens = (test_df['event_code'] == 0).sum() / n_test
obs_prepay = (test_df['event_code'] == 1).sum() / n_test
obs_default = (test_df['event_code'] == 2).sum() / n_test
print(f"\nObserved rates: censored={obs_cens:.3f}, prepay={obs_prepay:.3f}, default={obs_default:.3f}")

In [ ]:
# Cure fraction distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, values) in zip(axes, cure.items()):
    ax.hist(values, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'Cure fraction: {name}')
    ax.set_xlabel(f'pi_{name}(x)')
    ax.set_ylabel('Count')
    ax.axvline(values.mean(), color='red', linestyle='--', label=f'mean={values.mean():.3f}')
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cure_fraction_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# AUC_cure: ability to discriminate cured vs uncured borrowers
auc_overall = auc_cure(
    test_df['event_code'].values,
    test_df['duration'].values,
    cure['overall'],
    min_followup=120,
)
print(f"AUC_cure (overall, min_followup=120): {auc_overall:.4f}")

# Try different followup thresholds
for mf in [60, 84, 120, 150]:
    auc_val = auc_cure(
        test_df['event_code'].values,
        test_df['duration'].values,
        cure['overall'],
        min_followup=mf,
    )
    n_cured = ((test_df['event_code'] == 0) & (test_df['duration'] >= mf)).sum()
    print(f"  min_followup={mf:3d}: AUC={auc_val:.4f}  (n_cured={n_cured})")

## Model Evaluation

### Time-Dependent Concordance Index

In [ ]:
# Predict risk scores (CIF at evaluation times) for both events
test_durations = test_df['duration'].values
test_events = test_df['event_code'].values
X_test = test_df[FEATURE_COLS]

results_rows = []
for event_code, event_name in [(1, 'Prepay'), (2, 'Default')]:
    print(f"\n=== {event_name} (k={event_code}) ===")
    for t in EVAL_TIMES:
        risk_scores = model.predict_risk(X_test, event=event_code, time=t)
        c_idx, n_conc, n_comp = time_dependent_concordance_index(
            test_durations, test_events, risk_scores, t, event_code
        )
        bs = brier_score_competing_risks(
            test_durations, test_events,
            model.predict_cumulative_incidence(X_test, event=event_code, times=np.array([t]))[:, 0],
            t, event_code
        )
        print(f"  t={t:3d}: C-index={c_idx:.4f}  BS={bs:.4f}  (n_comparable={n_comp:,})")
        results_rows.append({
            'Event': event_name, 'Horizon': t,
            'C-index': c_idx, 'Brier Score': bs,
        })

results_df = pd.DataFrame(results_rows)

# Add means
for event_name in ['Prepay', 'Default']:
    mask = results_df['Event'] == event_name
    mean_c = results_df.loc[mask, 'C-index'].mean()
    mean_bs = results_df.loc[mask, 'Brier Score'].mean()
    results_rows.append({'Event': event_name, 'Horizon': 'Mean', 'C-index': mean_c, 'Brier Score': mean_bs})

results_df = pd.DataFrame(results_rows)
print("\n=== Summary ===")
print(results_df.to_string(index=False))

### Integrated Brier Score

In [ ]:
# Integrated Brier Score over [1, 72] months
ibs_grid = np.arange(1, 73, dtype='float32')

for event_code, event_name in [(1, 'Prepay'), (2, 'Default')]:
    cif_at_grid = model.predict_cumulative_incidence(
        X_test, event=event_code, times=ibs_grid
    )
    ibs = integrated_brier_score(
        test_durations, test_events,
        cif_at_grid, ibs_grid,
        event_of_interest=event_code,
    )
    print(f"IBS ({event_name}, t=1..72): {ibs:.6f}")

## Survival and CIF Curves

In [ ]:
# Select example loans: one low-risk, one medium, one high-risk (by FICO)
test_sorted = test_df.sort_values('fico_score')
idx_low = test_sorted.index[len(test_sorted) // 10]       # low FICO
idx_med = test_sorted.index[len(test_sorted) // 2]        # median FICO
idx_high = test_sorted.index[9 * len(test_sorted) // 10]  # high FICO
example_indices = [idx_low, idx_med, idx_high]
example_labels = ['Low FICO', 'Mid FICO', 'High FICO']

times_grid = np.linspace(1, 180, 180).astype('float32')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Overall survival
ax = axes[0]
for idx, label in zip(example_indices, example_labels):
    X_ex = test_df.loc[[idx], FEATURE_COLS]
    surv = model.predict_survival(X_ex, times=times_grid)
    fico = test_df.loc[idx, 'fico_score']
    ax.plot(times_grid, surv[0], label=f'{label} (FICO={fico:.0f})')
ax.set_xlabel('Months')
ax.set_ylabel('S(t)')
ax.set_title('Overall Survival')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Prepayment CIF
ax = axes[1]
for idx, label in zip(example_indices, example_labels):
    X_ex = test_df.loc[[idx], FEATURE_COLS]
    cif = model.predict_cumulative_incidence(X_ex, event=1, times=times_grid)
    ax.plot(times_grid, cif[0], label=label)
ax.set_xlabel('Months')
ax.set_ylabel('CIF_prepay(t)')
ax.set_title('Prepayment CIF')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Default CIF
ax = axes[2]
for idx, label in zip(example_indices, example_labels):
    X_ex = test_df.loc[[idx], FEATURE_COLS]
    cif = model.predict_cumulative_incidence(X_ex, event=2, times=times_grid)
    ax.plot(times_grid, cif[0], label=label)
ax.set_xlabel('Months')
ax.set_ylabel('CIF_default(t)')
ax.set_title('Default CIF')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'survival_cif_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Verify CIF consistency: S(t) + CIF_1(t) + CIF_2(t) should equal 1
X_check = test_df[FEATURE_COLS].head(100)
t_check = np.array([24, 48, 72, 120, 180], dtype='float32')
surv = model.predict_survival(X_check, times=t_check)
cif1 = model.predict_cumulative_incidence(X_check, event=1, times=t_check)
cif2 = model.predict_cumulative_incidence(X_check, event=2, times=t_check)

total = surv + cif1 + cif2
print("S(t) + CIF_1(t) + CIF_2(t) should be close to 1.0:")
for i, t in enumerate(t_check):
    print(f"  t={t:5.0f}: mean={total[:, i].mean():.6f}, max_deviation={np.abs(total[:, i] - 1.0).max():.6f}")

## Orthogonalized Model (Deep-PTCM-Ort)

The orthogonalized variant decomposes the predictor as $g_k(\mathbf{x}) = \mathbf{w}_k^\top\mathbf{x} + b_k + \text{ort}_k(\mathbf{x})$, where the nonlinear component is constrained to be orthogonal to the linear span of $\mathbf{X}$. This yields interpretable linear coefficients while retaining full DNN flexibility.

In [ ]:
# Train orthogonalized variant
PTCM_ORT_PARAMS = {**PTCM_PARAMS, 'orthogonalize': True}

model_ort = fit_deep_ptcm_competing_risks(
    df=train_df,
    feature_cols=FEATURE_COLS,
    duration_col='duration',
    event_col='event_code',
    event_types=[1, 2],
    val_df=val_df,
    **PTCM_ORT_PARAMS,
)

print(f"\nOrthogonalized model training complete.")
print(f"  Epochs run: {len(model_ort.history_['train_loss'])}")
print(f"  Best val loss: {min(model_ort.history_['val_loss']):.4f}")

In [ ]:
# Extract and display linear coefficients
coefs = model_ort.get_linear_coefficients()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (event_name, coef_df) in zip(axes, coefs.items()):
    # Exclude intercept for the bar plot
    plot_df = coef_df[coef_df['feature'] != '(intercept)'].copy()
    colors = ['#e74c3c' if c > 0 else '#3498db' for c in plot_df['coefficient']]
    ax.barh(plot_df['feature'], plot_df['coefficient'], color=colors)
    ax.set_title(f'Linear coefficients: {event_name}')
    ax.set_xlabel('Coefficient (on log-theta scale)')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'orthogonalized_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

# Print coefficients
for event_name, coef_df in coefs.items():
    print(f"\n{event_name}:")
    print(coef_df.to_string(index=False))

In [ ]:
# Compare Deep-PTCM vs Deep-PTCM-Ort on test set
print("=== Deep-PTCM vs Deep-PTCM-Ort ===\n")
comparison_rows = []

for model_obj, model_name in [(model, 'Deep-PTCM'), (model_ort, 'Deep-PTCM-Ort')]:
    for event_code, event_name in [(1, 'Prepay'), (2, 'Default')]:
        for t in EVAL_TIMES:
            risk = model_obj.predict_risk(X_test, event=event_code, time=t)
            c_idx, _, _ = time_dependent_concordance_index(
                test_durations, test_events, risk, t, event_code
            )
            comparison_rows.append({
                'Model': model_name, 'Event': event_name,
                'Horizon': t, 'C-index': c_idx,
            })

comp_df = pd.DataFrame(comparison_rows)
pivot = comp_df.pivot_table(
    index=['Event', 'Horizon'], columns='Model', values='C-index'
)
print(pivot.round(4).to_string())

## Ablation: Deeper Cause-Specific Heads

The paper uses a single linear output per cause (since it models only default). With competing risks, prepayment and default have distinct risk profiles that may require more cause-specific capacity. We test adding a hidden layer of 32 units to each head (`head_layers=[32]`) and compare all four variants.

In [ ]:
# Train Deep-PTCM with deeper heads (head_layers=[32])
PTCM_HEAD32_PARAMS = {**PTCM_PARAMS, 'head_layers': [32]}

model_head32 = fit_deep_ptcm_competing_risks(
    df=train_df,
    feature_cols=FEATURE_COLS,
    duration_col='duration',
    event_col='event_code',
    event_types=[1, 2],
    val_df=val_df,
    **PTCM_HEAD32_PARAMS,
)

print(f"\nDeep-PTCM (head=[32]) training complete.")
print(f"  Epochs run: {len(model_head32.history_['train_loss'])}")
print(f"  Best val loss: {min(model_head32.history_['val_loss']):.4f}")

# Train Deep-PTCM-Ort with deeper heads
PTCM_ORT_HEAD32_PARAMS = {**PTCM_PARAMS, 'head_layers': [32], 'orthogonalize': True}

model_ort_head32 = fit_deep_ptcm_competing_risks(
    df=train_df,
    feature_cols=FEATURE_COLS,
    duration_col='duration',
    event_col='event_code',
    event_types=[1, 2],
    val_df=val_df,
    **PTCM_ORT_HEAD32_PARAMS,
)

print(f"\nDeep-PTCM-Ort (head=[32]) training complete.")
print(f"  Epochs run: {len(model_ort_head32.history_['train_loss'])}")
print(f"  Best val loss: {min(model_ort_head32.history_['val_loss']):.4f}")

In [ ]:
# Compare all four variants
print("=== 4-Model Comparison: head_layers=[] vs head_layers=[32] ===\n")

all_models = [
    (model,           'Deep-PTCM'),
    (model_ort,       'Deep-PTCM-Ort'),
    (model_head32,    'Deep-PTCM (head=[32])'),
    (model_ort_head32,'Deep-PTCM-Ort (head=[32])'),
]

all_rows = []
for model_obj, model_name in all_models:
    for event_code, event_name in [(1, 'Prepay'), (2, 'Default')]:
        for t in EVAL_TIMES:
            risk = model_obj.predict_risk(X_test, event=event_code, time=t)
            c_idx, _, _ = time_dependent_concordance_index(
                test_durations, test_events, risk, t, event_code
            )
            cif_t = model_obj.predict_cumulative_incidence(
                X_test, event=event_code, times=np.array([t])
            )[:, 0]
            bs = brier_score_competing_risks(
                test_durations, test_events, cif_t, t, event_code
            )
            all_rows.append({
                'Model': model_name, 'Event': event_name,
                'Horizon': t, 'C-index': c_idx, 'Brier Score': bs,
            })

all_comp_df = pd.DataFrame(all_rows)

# C-index comparison
print("--- C-index ---")
c_pivot = all_comp_df.pivot_table(
    index=['Event', 'Horizon'], columns='Model', values='C-index'
)
print(c_pivot.round(4).to_string())

# Brier score comparison
print("\n--- Brier Score ---")
bs_pivot = all_comp_df.pivot_table(
    index=['Event', 'Horizon'], columns='Model', values='Brier Score'
)
print(bs_pivot.round(6).to_string())

# Mean metrics per model
print("\n--- Mean C-index by Model and Event ---")
mean_c = all_comp_df.groupby(['Model', 'Event'])['C-index'].mean().unstack()
print(mean_c.round(4).to_string())

# AUC_cure comparison
print("\n--- AUC_cure (min_followup=120) ---")
for model_obj, model_name in all_models:
    cure_pred = model_obj.predict_cure_fraction(X_test)
    auc_val = auc_cure(
        test_events, test_durations, cure_pred['overall'], min_followup=120
    )
    print(f"  {model_name:30s}: {auc_val:.4f}")

## Feature Importance (Permutation)

In [ ]:
# Permutation importance: shuffle each feature and measure C-index drop
def permutation_importance(model_obj, X_df, durations, events, feature_cols,
                           event_code, eval_time=72, n_repeats=5, seed=42):
    """Compute permutation importance for each feature."""
    rng = np.random.RandomState(seed)
    
    # Baseline C-index
    base_risk = model_obj.predict_risk(X_df, event=event_code, time=eval_time)
    base_c, _, _ = time_dependent_concordance_index(
        durations, events, base_risk, eval_time, event_code
    )
    
    importances = {}
    for feat in feature_cols:
        drops = []
        for _ in range(n_repeats):
            X_perm = X_df.copy()
            X_perm[feat] = rng.permutation(X_perm[feat].values)
            perm_risk = model_obj.predict_risk(X_perm, event=event_code, time=eval_time)
            perm_c, _, _ = time_dependent_concordance_index(
                durations, events, perm_risk, eval_time, event_code
            )
            drops.append(base_c - perm_c)
        importances[feat] = np.mean(drops)
    
    return pd.DataFrame([
        {'feature': f, 'importance': v}
        for f, v in importances.items()
    ]).sort_values('importance', ascending=False)

# Compute for both events
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (event_code, event_name) in zip(axes, [(1, 'Prepay'), (2, 'Default')]):
    imp_df = permutation_importance(
        model, test_df[FEATURE_COLS], test_durations, test_events,
        FEATURE_COLS, event_code=event_code,
    )
    ax.barh(imp_df['feature'], imp_df['importance'], color='steelblue')
    ax.set_title(f'Feature Importance: {event_name} (t=72)')
    ax.set_xlabel('C-index drop (permutation)')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='x')
    
    print(f"\n{event_name}:")
    print(imp_df.to_string(index=False))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Models

In [ ]:
# Save both models
model.save(str(MODELS_DIR / 'deep_ptcm.pt'))
model_ort.save(str(MODELS_DIR / 'deep_ptcm_ort.pt'))
print(f"Saved models to {MODELS_DIR}")

# Verify round-trip
model_loaded = CompetingRisksDeepPTCM.load(str(MODELS_DIR / 'deep_ptcm.pt'))
risk_orig = model.predict_risk(X_test, event=1, time=72)
risk_loaded = model_loaded.predict_risk(X_test, event=1, time=72)
print(f"Load verification - max difference: {np.abs(risk_orig - risk_loaded).max():.1e}")

## Summary

### Deep-PTCM Results

| Metric | Description |
|--------|-------------|
| **C-index** | Time-dependent concordance at t = 24, 48, 72 months |
| **Brier Score** | Calibration at each horizon |
| **IBS** | Integrated Brier Score over t = 1..72 |
| **AUC_cure** | Discrimination between cured and uncured borrowers |

### Key Findings

- The Deep-PTCM provides **cure fractions** for each competing risk, which other models cannot
- The **orthogonalized variant** yields interpretable linear coefficients while maintaining DNN flexibility
- Cure fractions for default should be high (most borrowers never default), while prepayment cure fractions capture the subset of borrowers who hold their mortgage to maturity

### Comparison with Other Models

The C-index results can be directly compared with:
- **Cause-Specific Cox** (notebook 05)
- **DeepHit** (notebook 08)
- **Random Survival Forest** (notebook 07)
- **Sadhwani NN** (notebook 18)

Note: Those models use time-varying panel data, while Deep-PTCM uses static loan-level features only. The comparison reveals the relative value of temporal vs. cure-model structure.

In [ ]:
# Final summary table
print("=" * 70)
print("DEEP-PTCM COMPETING RISKS - FINAL RESULTS")
print("=" * 70)
print(f"\nDataset: blumenstock_dataset2.parquet")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Features: {FEATURE_COLS}")
print(f"\n{results_df.to_string(index=False)}")
print(f"\nAUC_cure (overall, min_followup=120): {auc_overall:.4f}")
print("=" * 70)